# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [10]:
# 1
print("Строк и столбцов", df.shape)
print(df.dtypes)
print("Пропусков в delivery_days", len(df) - df["delivery_days"].count())
print("city:", df["city"].nunique())
print("category:", df["category"].nunique())
print("status:", df["status"].nunique())
print("customer_id:", df["customer_id"].nunique())
print(df["city"].value_counts())
print("Строк", len(df), "разных order_id", df["order_id"].nunique())

Строк и столбцов (5463, 8)
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object
Пропусков в delivery_days 160
city: 14
category: 5
status: 2
customer_id: 2003
city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64
Строк 5463 разных order_id 5283



Проблемы в том что цена записана текстом, разные вариации записи названия города а также есть повторения заказов


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [13]:
# 2
x = df.loc[paid & not_test, "amount"]

print("Среднее", round(x.mean()))
print("Медиана", x.median())
print("25 перцентиль", x.quantile(0.25))
print("75 перцентиль", x.quantile(0.75))

Среднее 14672
Медиана 10300.0
25 перцентиль 5880.0
75 перцентиль 18690.0


Честнее сказать медиану потому что среднее завыешает небольшое количество дорогих покупок

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [17]:
# 3
cancel = df["status"] == "отменён"
ALM = ["Алматы", " Алматы", "алматы", "АЛМАТЫ", "Almaty"]
sh_mask = df["city"].isin(SH)
alm_mask = df["city"].isin(ALM)
all_share = cancel.mean()
sh_share = cancel[sh_mask].mean()
alm_share = cancel[alm_mask].mean()

print("Все заказы", round(all_share * 100, 1))
print("Шымкент", round(sh_share * 100, 1))
print("Алматы", round(alm_share * 100, 1))
print("Шымкент / Алматы", round(sh_share / alm_share, 1))

Все заказы 10.2
Шымкент 20.4
Алматы 7.9
Шымкент / Алматы 2.6


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [18]:
# 4
good = paid & not_test
sh_sep = df.loc[good & sh_mask & sep, "amount"].sum()
sh_oct = df.loc[good & sh_mask & octo, "amount"].sum()
alm_sep = df.loc[good & alm_mask & sep, "amount"].sum()
alm_oct = df.loc[good & alm_mask & octo, "amount"].sum()
sh_change = (sh_oct - sh_sep) / sh_sep * 100
alm_change = (alm_oct - alm_sep) / alm_sep * 100

print("Шымкент до", sh_sep, "после", sh_oct, "|", round(sh_change, 1))
print("Алмата до", alm_sep, "после", alm_oct, "|", round(alm_change, 1))
print("Падение в Шымкенте сильнее в", round(sh_change / alm_change, 1))

Шымкент до 4949800.0 после 4044010.0 | -18.3
Алмата до 15523940.0 после 14643020.0 | -5.7
Падение в Шымкенте сильнее в 3.2


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [25]:
# 5
cancel_sh = df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]
print("Отмены в Шымкенте", len(cancel_sh))

Отмены в Шымкенте 153


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [26]:
# 6
print(df["customer_id"].value_counts().head())

top = df["customer_id"].value_counts().index[0]
print()
print("Больше всего заказов у", top)

top_orders = df.loc[df["customer_id"] == top, ["order_id", "order_date", "city", "amount", "status"]]
print(top_orders)

print()
print("Заказов", len(top_orders), "| разных order_id", top_orders["order_id"].nunique())

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64

Больше всего заказов у C01818
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

Заказов 11 | разных order_id 10


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [30]:
# 7
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1
border = q3 + 1.5 * iqr

big = paid & not_test & (df["amount"] > border)
print("Заказов выше границы:", (x > border).sum())
print("Доля", round((x > border).mean() * 100, 1))
print(df.loc[big, "category"].value_counts())
print("Граница", round(x.mean() + 3 * x.std()))

Заказов выше границы: 320
Доля 6.5
category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64
Граница 54420


## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8
